# Metrics

> What communication cost: transmissions, losses, bits, latency, energy.

In [ ]:
#| default_exp metrics

In [ ]:
#| export

from __future__ import annotations

from dataclasses import dataclass

from comm_core.core import Transmission, TransmissionResult




In [ ]:
#| export

@dataclass
class CommunicationMetrics:

    transmissions: int = 0
    successful: int = 0
    failed: int = 0

    bits_sent: int = 0
    bits_received: int = 0

    total_latency: float = 0.0
    total_energy: float = 0.0

    @property
    def packet_loss(self) -> float:
        if self.transmissions == 0:
            return 0.0

        return self.failed / self.transmissions

    @property
    def average_latency(self) -> float:
        if self.successful == 0:
            return 0.0

        return self.total_latency / self.successful



In [ ]:
#| export
class MetricsCollector:
    """Accumulates `CommunicationMetrics` over the transmissions' results.

    The copies of a broadcast (`metadata['broadcast']`) sent in the same round are one emission: they count as
    transmissions, deliveries and losses one by one, but their bits and energy once (the most of the copies')."""

    def __init__(self):
        self.metrics = CommunicationMetrics()
        self._emitted = {}                             # (broadcast, round) -> (bits, energy) charged so far

    def record(
        self,
        transmission: Transmission,
        result: TransmissionResult,
    ):

        if not isinstance(result, TransmissionResult):
            raise TypeError(f"expected a TransmissionResult, got {type(result).__name__}")

        m = self.metrics

        m.transmissions += 1

        bits = result.size_bits
        energy = result.energy

        if result.success:
            m.successful += 1
            m.bits_received += bits
            m.total_latency += result.latency

        else:
            m.failed += 1

        broadcast = transmission.message.metadata.get('broadcast')

        if broadcast is not None:
            # another copy of the same emission: charge only what it costs beyond the copies before it
            if any(start != transmission.start_time for _, start in self._emitted):
                self._emitted.clear()                  # a new round: earlier rounds' emissions are over
            key = (broadcast, transmission.start_time)
            charged_bits, charged_energy = self._emitted.get(key, (0, 0.0))
            self._emitted[key] = (max(charged_bits, bits), max(charged_energy, energy))
            bits, energy = max(bits - charged_bits, 0), max(energy - charged_energy, 0.0)

        m.bits_sent += bits

        m.total_energy += energy

    def reset(self) -> None:
        "Start counting again."
        self.metrics = CommunicationMetrics()
        self._emitted = {}

    def summary(self) -> CommunicationMetrics:
        return self.metrics
    

### Tests

In [ ]:
from comm_core.core import Message, Transmission

In [ ]:
def result(message, start, success=True, bits=8, energy=1.0):
    t = Transmission(message=message, sender=message.sender, receiver=message.receiver, start_time=start)
    return t, TransmissionResult(success=success, transmission=t, size_bits=bits, energy=energy)

mc = MetricsCollector()
mc.record(*result(Message('a', 'b', 0), 0.0))
mc.record(*result(Message('a', 'c', 0), 0.0))
assert (mc.summary().bits_sent, mc.summary().total_energy) == (16, 2.0)          # unicasts: each its own cost
# a broadcast to three neighbours in one round: one emission, at its costliest copy
mc.reset()
for receiver, energy, success in (('b', 1.0, True), ('c', 3.0, False), ('d', 2.0, True)):
    mc.record(*result(Message('a', receiver, 0, metadata={'broadcast': 'x'}), 0.0, success, energy=energy))
s = mc.summary()
assert (s.transmissions, s.successful, s.failed, s.bits_sent, s.bits_received, s.total_energy) == (3, 2, 1, 8, 16, 3.0)
# its copy sent again in a later round is another emission
mc.record(*result(Message('a', 'c', 0, metadata={'broadcast': 'x'}), 1.0))
assert (mc.summary().bits_sent, mc.summary().total_energy) == (16, 4.0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()